<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/Precipitation_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade xee

In [ ]:
!pip install -U geemap

In [ ]:
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project = 'ee-grmntfrancis0',
     opt_url = 'https://earthengine-highvolume.googleapis.com')

In [ ]:
import geemap

In [ ]:
m = geemap.Map()
m

In [ ]:
roi = m.draw_last_feature.geometry()
roi

In [ ]:
border = (
    ee.FeatureCollection("FAO/GAUL/2015/level0")
    .filterBounds(roi)
    .geometry().simplify(100)
)

In [ ]:
m.addLayer(border, {}, "border")

In [ ]:
gpm = (
    ee.ImageCollection("UCSB-CHC/CHIRPS/V3/DAILY_RNL")
    .filterDate('2015-01-01','2025-12-31')
    .select('precipitation')
    .map(
        lambda img: img.clip(border).copyProperties(img, ['system:time_start'])
    )
)

gpm.getInfo()

In [ ]:
import xarray as xr

In [ ]:
pr = xr.open_dataset(
    gpm,
    engine = 'ee',
    crs = 'EPSG:4326',
    scale = 0.1,
    geometry = border
)

In [ ]:
pr = pr.sortby('time')

In [ ]:
pr

In [ ]:
import numpy as np

In [ ]:
pr_annual = pr.resample(time = 'YE').sum('time')

In [ ]:
pr_annual = xr.where(pr_annual == 0,  np.nan, pr_annual)

In [ ]:
pr_annual.precipitation.plot.contourf(
    x = 'lon',
    y = 'lat',
    col = 'time',
    col_wrap = 5,
    robust = True,
    level = 20,
    cmap = "turbo_r"

)

annual_change = pr_annual.diff('time')


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.savefig('pr_change.png', dpi = 360, bbox_inches = 'tight')
annual_change.precipitation.plot.contourf(
    x = 'lon',
    y = 'lat',
    col = 'time',
    col_wrap = 5,
    robust = True,
    levels = 10,
    cmap = "turbo_r"

)
plt.savefig('pr_change.png', dpi = 360, bbox_inches = 'tight')


In [ ]:
mean_change = annual_change.mean(dim = 'time')

mean_change.precipitation.plot.contourf(
    x = 'lon',
    y = 'lat',
    robust =True,
    levels = 10,
    cmap = "turbo"
    aspect = "equal"
)

In [ ]:
pr_annual_spatial_mean = pr_annual.mean(dim=['lat', 'lon'])


In [ ]:
pr_annual_smoothed = pr_annual_spatial_mean.rolling(time=5, center=True).mean().fillna(pr_annual_spatial_mean)

In [ ]:
plt.figure(figsize=(12, 6))
pr_annual_spatial_mean.precipitation.plot(label='Original Annual Precipitation', color='blue', linestyle='-')
pr_annual_smoothed.precipitation.plot.line(label='Smoothed Annual Precipitation (5-year rolling mean)', color='red', linestyle='--')
plt.title('Annual Mean Precipitation Time Series with Smoothing')
plt.xlabel('Year')
plt.ylabel('Mean Precipitation')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
pr_annual_spatial_mean.precipitation.plot(label='Original Annual Precipitation', color='blue', linestyle='-')

In [ ]:
pr_annual_smoothed.precipitation.plot.line(label='Smoothed Annual Precipitation (5-year rolling mean)', color='red', linestyle='--')